# Task 2 Model 1: Random Forest

This notebook loads the prepared data exported by Notebook 1, engineers visual features,
trains Random Forest, and exports only this model's artefacts.


## How to Run

Run `01_task2_setup.ipynb` once before this notebook. This notebook does not repeat the split,
image decoding, normalisation, or baseline work.


## 1. Setup


In [ ]:
%matplotlib inline
import joblib
import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.ensemble import RandomForestClassifier
from sklearn.inspection import permutation_importance


In [ ]:
from pathlib import Path
import sys

REPO_ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
                  if (p / "pyproject.toml").exists()), None)
if REPO_ROOT is None:
    raise FileNotFoundError("Could not find the repository root containing pyproject.toml")
sys.path.insert(0, str(REPO_ROOT))

from src.task2_utils import (
    ensure_task2_directories, per_class_table, prepared_namespace,
    random_forest_paths, restore_random_forest_checkpoint,
    result_frame, save_random_forest_checkpoint,
)
ensure_task2_directories()


### 1.1 Random Forest configuration


In [ ]:
QUICK_RUN = True
RANDOM_STATE = 42
FEATURE_N_JOBS = -1
RF_N_ESTIMATORS = 100 if QUICK_RUN else 500
RF_MAX_DEPTH = None
RF_MIN_SAMPLES_LEAF = 2
RF_MAX_FEATURES = "sqrt"
MODEL_CONFIG = {
    "architecture": "random_forest", "n_estimators": RF_N_ESTIMATORS,
    "max_depth": RF_MAX_DEPTH, "min_samples_leaf": RF_MIN_SAMPLES_LEAF,
    "max_features": RF_MAX_FEATURES, "class_weight": "balanced_subsample",
    "random_state": RANDOM_STATE,
}
MODEL_PATH, SCORES_PATH = random_forest_paths()


## 2. Load Prepared Data from Notebook 1


In [ ]:
data = prepared_namespace()
assert data.config.get("quick_run") == QUICK_RUN, (
    "QUICK_RUN does not match Notebook 1's prepared cache. "
    "Set the same value in both notebooks and rerun Notebook 1 if needed."
)
print(f"Loaded {len(data.train_frame):,} train and {len(data.validation_frame):,} validation rows")


## 3. Train Random Forest


In [ ]:
X_train_features = data.x_train_features
X_val_features = data.x_val_features
print("Loaded feature matrices:", X_train_features.shape, X_val_features.shape)

restored = restore_random_forest_checkpoint(data, MODEL_CONFIG)
random_forest = RandomForestClassifier(
    n_estimators=RF_N_ESTIMATORS, max_depth=RF_MAX_DEPTH,
    min_samples_leaf=RF_MIN_SAMPLES_LEAF, max_features=RF_MAX_FEATURES,
    class_weight="balanced_subsample", n_jobs=-1, random_state=RANDOM_STATE,
)
if restored is None:
    random_forest.fit(X_train_features, data.y_train)
    rf_scores = random_forest.predict_proba(X_val_features)
    save_random_forest_checkpoint(random_forest, rf_scores, data, MODEL_CONFIG)
else:
    random_forest, rf_scores = restored
    print("Restored compatible Random Forest checkpoint; fitting skipped")
rf_pred = random_forest.classes_[rf_scores.argmax(axis=1)]
display(result_frame(data.y_val, rf_pred, rf_scores, "Random Forest"))


### 3.1 Feature evidence


In [ ]:
importance = permutation_importance(
    random_forest, X_val_features, data.y_val, scoring="f1_macro",
    n_repeats=3 if QUICK_RUN else 8, random_state=RANDOM_STATE, n_jobs=-1,
)
feature_evidence = pd.DataFrame({
    "feature_index": np.arange(X_train_features.shape[1]),
    "importance_mean": importance.importances_mean,
    "importance_std": importance.importances_std,
}).sort_values("importance_mean", ascending=False)
display(feature_evidence.head(20))
display(per_class_table(data.y_val, rf_pred, data.classes))


## 4. Export Random Forest Results


In [ ]:
feature_path = MODEL_PATH.parent / "random_forest_feature_importance.csv"
feature_evidence.to_csv(feature_path, index=False)
print("Saved model:", MODEL_PATH)
print("Saved validation scores:", SCORES_PATH)
print("Run fingerprint:", data.fingerprint)
